# Machine Learning: LightGBM & CatBoost (n1)

**Objective**: Train LightGBM and CatBoost classifiers combining:
1. **TF-IDF embeddings** (900-dim from strict text) - GENERATED
2. **PhoBERT v2 Pretrained** (768-dim Layer 12 from loose text) - EXTRACTED
3. **ALL additional hand-crafted features** - EXTRACTED FROM DATASET

**Strategy**:
- Generate TF-IDF embeddings on-the-fly
- Extract PhoBERT embeddings from pre-trained model directly
- Extract ALL non-text columns as hand-crafted features
- Combine via concatenation
- Train LightGBM and CatBoost with StandardScaler + balanced weights
- Evaluate on validation set
- Save model + scaler + feature importance

## 1. Import Libraries & Device Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib
import torch
import gc
from tqdm import tqdm
warnings.filterwarnings('ignore')

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from transformers import AutoTokenizer, AutoModel

# Setup
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Libraries imported successfully")
print(f"   Device: {device}")

✅ Libraries imported successfully
   Device: cuda


## 2. Load Data

In [2]:
print("="*80)
print("LOADING DATA")
print("="*80)

# Load train/val sets
train_path = '../../../data/splited/train_set.csv'
val_path = '../../../data/splited/val_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

print(f"\nData loaded:")
print(f"  Train: {df_train.shape[0]} samples × {df_train.shape[1]} columns")
print(f"  Val:   {df_val.shape[0]} samples × {df_val.shape[1]} columns")
print(f"\nAvailable columns ({len(df_train.columns)}):")
for i, col in enumerate(df_train.columns, 1):
    print(f"  {i:2d}. {col}")

# Extract labels
y_train = df_train['label'].values
y_val = df_val['label'].values

print(f"\nLabels:")
print(f"  Train: {(y_train == 0).sum()} fake / {(y_train == 1).sum()} real")
print(f"  Val:   {(y_val == 0).sum()} fake / {(y_val == 1).sum()} real")

LOADING DATA

Data loaded:
  Train: 3788 samples × 28 columns
  Val:   474 samples × 28 columns

Available columns (28):
   1. id
   2. post_message
   3. label
   4. num_char
   5. num_emoji
   6. num_url
   7. num_hashtag
   8. num_like
   9. num_cmt
  10. num_share
  11. text_strict
  12. text_loose
  13. feat_word_count
  14. feat_exclamation
  15. feat_question
  16. feat_uppercase_words
  17. feat_uppercase_ratio
  18. feat_punctuation_ratio
  19. feat_engagement_total
  20. feat_like_ratio
  21. feat_comment_ratio
  22. feat_like_per_char
  23. feat_hour_sin
  24. feat_hour_cos
  25. feat_day_sin
  26. feat_day_cos
  27. feat_month_sin
  28. feat_month_cos

Labels:
  Train: 3143 fake / 645 real
  Val:   393 fake / 81 real


## 3. Generate TF-IDF Embeddings (900-dim)

In [3]:
print("\n" + "="*80)
print("GENERATING TF-IDF EMBEDDINGS (900-dim)")
print("="*80)

# Extract strict text for TF-IDF
texts_train_strict = df_train['text_strict'].fillna('').tolist()
texts_val_strict = df_val['text_strict'].fillna('').tolist()

print(f"\nGenerating TF-IDF from strict text...")
print(f"  Train texts: {len(texts_train_strict)}")
print(f"  Val texts: {len(texts_val_strict)}")

# Create FULL TF-IDF vectorizer (same config as n0_TFIDF_statistical_methods)
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_df=0.95,
    min_df=2,
    sublinear_tf=True
)

# Fit on train, transform both
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)
X_val_tfidf_full = tfidf.transform(texts_val_strict)

print(f"\nFull TF-IDF vocabulary size: {X_train_tfidf_full.shape[1]}")

# Apply TruncatedSVD to reduce to 900 dims
svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)
X_val_tfidf = svd.transform(X_val_tfidf_full)

variance_retained = svd.explained_variance_ratio_.sum()

print(f"\n✅ TF-IDF embeddings generated:")
print(f"   Train: {X_train_tfidf.shape}")
print(f"   Val: {X_val_tfidf.shape}")
print(f"   Variance retained: {variance_retained:.2%}")


GENERATING TF-IDF EMBEDDINGS (900-dim)

Generating TF-IDF from strict text...
  Train texts: 3788
  Val texts: 474

Full TF-IDF vocabulary size: 57103

✅ TF-IDF embeddings generated:
   Train: (3788, 900)
   Val: (474, 900)
   Variance retained: 49.05%


## 4. Extract PhoBERT Embeddings (Layer 12)

In [4]:
print("\n" + "="*80)
print("EXTRACTING PHOBERT EMBEDDINGS (Layer 12 - Pretrained)")
print("="*80)

# Extract loose text for PhoBERT
texts_train_loose = df_train['text_loose'].fillna('').tolist()
texts_val_loose = df_val['text_loose'].fillna('').tolist()

def extract_phobert_embeddings(texts, batch_size=16):
    """
    Extract [CLS] token embeddings from Layer 12 of PhoBERT v2 Pretrained model.
    Returns: numpy array of shape (num_texts, 768)
    """
    print(f"    Loading model...")
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
    model = AutoModel.from_pretrained('vinai/phobert-base-v2').to(device).eval()
    
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="    Extracting"):
            batch = texts[i:i+batch_size]
            
            # Tokenize
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Forward pass
            outputs = model(**inputs, output_hidden_states=True)
            
            # Extract [CLS] from Layer 12 (encoder output = last_hidden_state)
            cls_tokens = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            
            del outputs, inputs
            torch.cuda.empty_cache()
    
    # Clean up
    model.to('cpu')
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    return np.array(embeddings)

print(f"\nExtracting PhoBERT v2 Pretrained (Layer 12) from loose text...")
print(f"  Train texts: {len(texts_train_loose)}")
print(f"  Val texts: {len(texts_val_loose)}")

X_train_phobert = extract_phobert_embeddings(texts_train_loose, batch_size=16)
X_val_phobert = extract_phobert_embeddings(texts_val_loose, batch_size=16)

print(f"\n✅ PhoBERT embeddings extracted:")
print(f"   Train: {X_train_phobert.shape}")
print(f"   Val: {X_val_phobert.shape}")


EXTRACTING PHOBERT EMBEDDINGS (Layer 12 - Pretrained)

Extracting PhoBERT v2 Pretrained (Layer 12) from loose text...
  Train texts: 3788
  Val texts: 474
    Loading model...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
    Extracting: 100%|██████████| 237/237 [00:54<00:00,  4.33it/s]


    Loading model...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
    Extracting: 100%|██████████| 30/30 [00:07<00:00,  4.14it/s]



✅ PhoBERT embeddings extracted:
   Train: (3788, 768)
   Val: (474, 768)


## 5. Extract ALL Hand-Crafted Features

In [5]:
print("\n" + "="*80)
print("EXTRACTING ALL HAND-CRAFTED FEATURES (NUMERIC ONLY)")
print("="*80)

# Extract NUMERIC hand-crafted features only (exclude text columns, id, label, and non-numeric)
exclude_cols = {'id', 'label', 'text_strict', 'text_loose', 'post_message'}
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols 
                if col not in exclude_cols and col.strip() != '' and not col.startswith('Unnamed')]

print(f"\nHand-crafted NUMERIC features found: {len(feature_cols)}")
if feature_cols:
    print(f"\nFeature list:\"")
    for i, feat in enumerate(feature_cols, 1):
        dtype = df_train[feat].dtype
        print(f"  {i:2d}. {feat:30s} [{dtype}]")
    
    # Extract features and fill NaN with 0
    X_train_features = df_train[feature_cols].fillna(0).values
    X_val_features = df_val[feature_cols].fillna(0).values
    
    print(f"\n✅ Hand-crafted features extracted:")
    print(f"   Train: {X_train_features.shape}")
    print(f"   Val: {X_val_features.shape}")
else:
    print(f"   ℹ️  No numeric hand-crafted features found!")
    print(f"   (Excluding: {exclude_cols})")
    X_train_features = None
    X_val_features = None


EXTRACTING ALL HAND-CRAFTED FEATURES (NUMERIC ONLY)

Hand-crafted NUMERIC features found: 23

Feature list:"
   1. num_char                       [int64]
   2. num_emoji                      [int64]
   3. num_url                        [int64]
   4. num_hashtag                    [int64]
   5. num_like                       [int64]
   6. num_cmt                        [int64]
   7. num_share                      [int64]
   8. feat_word_count                [int64]
   9. feat_exclamation               [int64]
  10. feat_question                  [int64]
  11. feat_uppercase_words           [int64]
  12. feat_uppercase_ratio           [float64]
  13. feat_punctuation_ratio         [float64]
  14. feat_engagement_total          [int64]
  15. feat_like_ratio                [float64]
  16. feat_comment_ratio             [float64]
  17. feat_like_per_char             [float64]
  18. feat_hour_sin                  [float64]
  19. feat_hour_cos                  [float64]
  20. feat_day_sin   

## 6. Combine All Features

In [6]:
print("\n" + "="*80)
print("COMBINING ALL FEATURES")
print("="*80)

# List all components
print(f"\nFeature dimensions:")
print(f"  TF-IDF (strict):        900")
print(f"  PhoBERT (loose):        768")

embeddings_list_train = [X_train_tfidf, X_train_phobert]
embeddings_list_val = [X_val_tfidf, X_val_phobert]

if X_train_features is not None:
    embeddings_list_train.append(X_train_features)
    embeddings_list_val.append(X_val_features)
    print(f"  Hand-crafted features: {X_train_features.shape[1]}")
    total_dims = 900 + 768 + X_train_features.shape[1]
else:
    total_dims = 900 + 768

# Concatenate all
X_train = np.hstack(embeddings_list_train)
X_val = np.hstack(embeddings_list_val)

print(f"\n{'─'*80}")
print(f"  TOTAL DIMENSIONS:       {total_dims}")
print(f"{'─'*80}")

print(f"\nCombined feature shapes:")
print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")


COMBINING ALL FEATURES

Feature dimensions:
  TF-IDF (strict):        900
  PhoBERT (loose):        768
  Hand-crafted features: 23

────────────────────────────────────────────────────────────────────────────────
  TOTAL DIMENSIONS:       1691
────────────────────────────────────────────────────────────────────────────────

Combined feature shapes:
  Train: (3788, 1691)
  Val:   (474, 1691)


## 7. Preprocessing & Scaling

In [7]:
print("\n" + "="*80)
print("DATA PREPROCESSING")
print("="*80)

# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f"\n✅ Data scaled using StandardScaler")
print(f"   Train - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")
print(f"   Val   - Mean: {X_val_scaled.mean():.6f}, Std: {X_val_scaled.std():.6f}")


DATA PREPROCESSING

✅ Data scaled using StandardScaler
   Train - Mean: -0.000000, Std: 1.000000
   Val   - Mean: -0.000678, Std: 0.815596


## 8. Train Models: LightGBM & CatBoost

In [8]:
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler

print("\n" + "="*80)
print("TRAINING MODELS")
print("="*80)

# Prepare embeddings features (TF-IDF + PhoBERT only)
scaler_emb = StandardScaler()
X_train_embeddings = np.hstack([X_train_tfidf, X_train_phobert])
X_val_embeddings = np.hstack([X_val_tfidf, X_val_phobert])
X_train_embeddings_scaled = scaler_emb.fit_transform(X_train_embeddings)
X_val_embeddings_scaled = scaler_emb.transform(X_val_embeddings)

# ============================================================================
# MODEL 1: LightGBM on embeddings only (TF-IDF + PhoBERT)
# ============================================================================
print(f"\n[MODEL 1] Training LightGBM on embeddings only ({X_train_embeddings_scaled.shape[1]} dims)...")
print(f"          (TF-IDF + PhoBERT)")

clf_model1 = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
clf_model1.fit(X_train_embeddings_scaled, y_train)
print(f"✅ Model 1 trained")

# ============================================================================
# MODEL 2: LightGBM on Combined (TF-IDF + PhoBERT + Hand-crafted)
# ============================================================================
print(f"\n[MODEL 2] Training LightGBM on combined features ({X_train_scaled.shape[1]} dims)...")
print(f"          (TF-IDF + PhoBERT + Hand-crafted)")

clf_model2 = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
clf_model2.fit(X_train_scaled, y_train)
print(f"✅ Model 2 trained")

# ============================================================================
# MODEL 3: Ensemble of sub-models (Embeddings + Features)
# ============================================================================
print(f"\n[MODEL 3] Ensemble approach:")

# Sub-model 3a: LightGBM on embeddings only
print(f"  [3a] Training LightGBM on embeddings only ({X_train_embeddings_scaled.shape[1]} dims)...")
print(f"       (TF-IDF + PhoBERT)")

clf_model3a = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
clf_model3a.fit(X_train_embeddings_scaled, y_train)
print(f"  ✅ Sub-model 3a trained")

# Sub-model 3b: LightGBM on Hand-crafted features only
if X_train_features is not None:
    print(f"  [3b] Training LightGBM on hand-crafted features ({X_train_features.shape[1]} dims)...")
    
    scaler_features = StandardScaler()
    X_train_features_scaled = scaler_features.fit_transform(X_train_features)
    X_val_features_scaled = scaler_features.transform(X_val_features)
    
    clf_model3b = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        is_unbalance=True,
        random_state=42,
        verbose=-1
    )
    clf_model3b.fit(X_train_features_scaled, y_train)
    print(f"  ✅ Sub-model 3b trained")
else:
    clf_model3b = None
    print(f"  ⚠️  No hand-crafted features for sub-model 3b")

# ============================================================================
# MODEL 4: Ensemble on embeddings only (LightGBM + CatBoost)
# ============================================================================
print(f"\n[MODEL 4] Ensemble approach (embeddings only):")

# Sub-model 4a: LightGBM on embeddings
print(f"  [4a] Training LightGBM on embeddings ({X_train_embeddings_scaled.shape[1]} dims)...")
print(f"       (TF-IDF + PhoBERT)")

clf_model4a = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
clf_model4a.fit(X_train_embeddings_scaled, y_train)
print(f"  ✅ Sub-model 4a trained")

# Sub-model 4b: CatBoost on embeddings
print(f"  [4b] Training CatBoost on embeddings ({X_train_embeddings_scaled.shape[1]} dims)...")
print(f"       (TF-IDF + PhoBERT)")

clf_model4b = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=7,
    auto_class_weights='Balanced',
    random_state=42,
    verbose=0
)
clf_model4b.fit(X_train_embeddings_scaled, y_train)
print(f"  ✅ Sub-model 4b trained")

# ============================================================================
# MODEL 5: Ensemble of different algorithms on combined features
# ============================================================================
print(f"\n[MODEL 5] Ensemble approach (different algorithms on combined):")

# Sub-model 5a: LightGBM on combined
print(f"  [5a] Training LightGBM on combined ({X_train_scaled.shape[1]} dims)...")
print(f"       (TF-IDF + PhoBERT + Hand-crafted)")

clf_model5a = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
clf_model5a.fit(X_train_scaled, y_train)
print(f"  ✅ Sub-model 5a trained")

# Sub-model 5b: CatBoost on combined
print(f"  [5b] Training CatBoost on combined ({X_train_scaled.shape[1]} dims)...")
print(f"       (TF-IDF + PhoBERT + Hand-crafted)")

clf_model5b = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=7,
    auto_class_weights='Balanced',
    random_state=42,
    verbose=0
)
clf_model5b.fit(X_train_scaled, y_train)
print(f"  ✅ Sub-model 5b trained")


TRAINING MODELS

[MODEL 1] Training LightGBM on embeddings only (1668 dims)...
          (TF-IDF + PhoBERT)
✅ Model 1 trained

[MODEL 2] Training LightGBM on combined features (1691 dims)...
          (TF-IDF + PhoBERT + Hand-crafted)
✅ Model 2 trained

[MODEL 3] Ensemble approach:
  [3a] Training LightGBM on embeddings only (1668 dims)...
       (TF-IDF + PhoBERT)
  ✅ Sub-model 3a trained
  [3b] Training LightGBM on hand-crafted features (23 dims)...
  ✅ Sub-model 3b trained

[MODEL 4] Ensemble approach (embeddings only):
  [4a] Training LightGBM on embeddings (1668 dims)...
       (TF-IDF + PhoBERT)
  ✅ Sub-model 4a trained
  [4b] Training CatBoost on embeddings (1668 dims)...
       (TF-IDF + PhoBERT)
  ✅ Sub-model 4b trained

[MODEL 5] Ensemble approach (different algorithms on combined):
  [5a] Training LightGBM on combined (1691 dims)...
       (TF-IDF + PhoBERT + Hand-crafted)
  ✅ Sub-model 5a trained
  [5b] Training CatBoost on combined (1691 dims)...
       (TF-IDF + PhoBERT 

## 9. Generate Predictions & Ensemble Results

In [9]:
print("\n" + "="*80)
print("EVALUATION & COMPARISON")
print("="*80)

# ============================================================================
# MODEL 1: LightGBM on embeddings only (TF-IDF + PhoBERT)
# ============================================================================
print(f"\n{'─'*80}")
print(f"[MODEL 1] LightGBM on Embeddings Only (TF-IDF + PhoBERT)")
print(f"{'─'*80}")

y_val_pred_m1 = clf_model1.predict(X_val_embeddings_scaled)
y_val_proba_m1 = clf_model1.predict_proba(X_val_embeddings_scaled)[:, 1]

m1_accuracy = accuracy_score(y_val, y_val_pred_m1)
m1_precision = precision_score(y_val, y_val_pred_m1, average='weighted')
m1_recall = recall_score(y_val, y_val_pred_m1, average='weighted')
m1_f1 = f1_score(y_val, y_val_pred_m1, average='weighted')
m1_auc = roc_auc_score(y_val, y_val_proba_m1)

print(f"Accuracy:  {m1_accuracy:.4f}")
print(f"Precision: {m1_precision:.4f}")
print(f"Recall:    {m1_recall:.4f}")
print(f"F1 Score:  {m1_f1:.4f}")
print(f"AUC-ROC:   {m1_auc:.4f}")

# ============================================================================
# MODEL 2: LightGBM on Combined Features (TF-IDF + PhoBERT + Hand-crafted)
# ============================================================================
print(f"\n{'─'*80}")
print(f"[MODEL 2] LightGBM on Combined Features (TF-IDF + PhoBERT + Hand-crafted)")
print(f"{'─'*80}")

y_val_pred_m2 = clf_model2.predict(X_val_scaled)
y_val_proba_m2 = clf_model2.predict_proba(X_val_scaled)[:, 1]

m2_accuracy = accuracy_score(y_val, y_val_pred_m2)
m2_precision = precision_score(y_val, y_val_pred_m2, average='weighted')
m2_recall = recall_score(y_val, y_val_pred_m2, average='weighted')
m2_f1 = f1_score(y_val, y_val_pred_m2, average='weighted')
m2_auc = roc_auc_score(y_val, y_val_proba_m2)

print(f"Accuracy:  {m2_accuracy:.4f}")
print(f"Precision: {m2_precision:.4f}")
print(f"Recall:    {m2_recall:.4f}")
print(f"F1 Score:  {m2_f1:.4f}")
print(f"AUC-ROC:   {m2_auc:.4f}")

# ============================================================================
# MODEL 3: Ensemble (Average of sub-models 3a and 3b)
# ============================================================================
print(f"\n{'─'*80}")
print(f"[MODEL 3] Ensemble (LightGBM(embeddings) avg LightGBM(features))")
print(f"{'─'*80}")

# Sub-model 3a predictions
y_val_proba_3a = clf_model3a.predict_proba(X_val_embeddings_scaled)[:, 1]

if clf_model3b is not None:
    # Sub-model 3b predictions
    y_val_proba_3b = clf_model3b.predict_proba(X_val_features_scaled)[:, 1]
    
    # Ensemble: Average probabilities from 3a and 3b
    y_val_proba_m3 = (y_val_proba_3a + y_val_proba_3b) / 2
    y_val_pred_m3 = (y_val_proba_m3 > 0.5).astype(int)
    
    m3_accuracy = accuracy_score(y_val, y_val_pred_m3)
    m3_precision = precision_score(y_val, y_val_pred_m3, average='weighted')
    m3_recall = recall_score(y_val, y_val_pred_m3, average='weighted')
    m3_f1 = f1_score(y_val, y_val_pred_m3, average='weighted')
    m3_auc = roc_auc_score(y_val, y_val_proba_m3)
    
    print(f"  Sub-model 3a (Embeddings): LightGBM on TF-IDF + PhoBERT")
    print(f"  Sub-model 3b (Features): LightGBM on Hand-crafted")
    print(f"  Combined: (prob_3a + prob_3b) / 2")
    print()
    print(f"Accuracy:  {m3_accuracy:.4f}")
    print(f"Precision: {m3_precision:.4f}")
    print(f"Recall:    {m3_recall:.4f}")
    print(f"F1 Score:  {m3_f1:.4f}")
    print(f"AUC-ROC:   {m3_auc:.4f}")
else:
    print(f"⚠️  Cannot create Model 3 - missing sub-model 3b")
    y_val_pred_m3 = None
    y_val_proba_m3 = None

# ============================================================================
# MODEL 4: Ensemble on embeddings only (LightGBM avg CatBoost)
# ============================================================================
print(f"\n{'─'*80}")
print(f"[MODEL 4] Ensemble on Embeddings (LightGBM avg CatBoost)")
print(f"{'─'*80}")

# Sub-model 4a predictions (LightGBM)
y_val_proba_4a = clf_model4a.predict_proba(X_val_embeddings_scaled)[:, 1]

# Sub-model 4b predictions (CatBoost)
y_val_proba_4b_raw = clf_model4b.predict_proba(X_val_embeddings_scaled)[:, 1]

# Ensemble: Average probabilities from 4a and 4b
y_val_proba_m4 = (y_val_proba_4a + y_val_proba_4b_raw) / 2
y_val_pred_m4 = (y_val_proba_m4 > 0.5).astype(int)

m4_accuracy = accuracy_score(y_val, y_val_pred_m4)
m4_precision = precision_score(y_val, y_val_pred_m4, average='weighted')
m4_recall = recall_score(y_val, y_val_pred_m4, average='weighted')
m4_f1 = f1_score(y_val, y_val_pred_m4, average='weighted')
m4_auc = roc_auc_score(y_val, y_val_proba_m4)

print(f"  Sub-model 4a: LightGBM on embeddings")
print(f"  Sub-model 4b: CatBoost on embeddings")
print(f"  Combined: (prob_4a + prob_4b) / 2")
print()
print(f"Accuracy:  {m4_accuracy:.4f}")
print(f"Precision: {m4_precision:.4f}")
print(f"Recall:    {m4_recall:.4f}")
print(f"F1 Score:  {m4_f1:.4f}")
print(f"AUC-ROC:   {m4_auc:.4f}")

# ============================================================================
# MODEL 5: Ensemble (Average of different algorithms on combined features)
# ============================================================================
print(f"\n{'─'*80}")
print(f"[MODEL 5] Ensemble (LightGBM avg CatBoost on combined)")
print(f"{'─'*80}")

# Sub-model 5a predictions (LightGBM)
y_val_proba_5a = clf_model5a.predict_proba(X_val_scaled)[:, 1]

# Sub-model 5b predictions (CatBoost)
y_val_proba_5b_raw = clf_model5b.predict_proba(X_val_scaled)[:, 1]

# Ensemble: Average probabilities from 5a and 5b
y_val_proba_m5 = (y_val_proba_5a + y_val_proba_5b_raw) / 2
y_val_pred_m5 = (y_val_proba_m5 > 0.5).astype(int)

m5_accuracy = accuracy_score(y_val, y_val_pred_m5)
m5_precision = precision_score(y_val, y_val_pred_m5, average='weighted')
m5_recall = recall_score(y_val, y_val_pred_m5, average='weighted')
m5_f1 = f1_score(y_val, y_val_pred_m5, average='weighted')
m5_auc = roc_auc_score(y_val, y_val_proba_m5)

print(f"  Sub-model 5a: LightGBM on combined")
print(f"  Sub-model 5b: CatBoost on combined")
print(f"  Combined: (prob_5a + prob_5b) / 2")
print()
print(f"Accuracy:  {m5_accuracy:.4f}")
print(f"Precision: {m5_precision:.4f}")
print(f"Recall:    {m5_recall:.4f}")
print(f"F1 Score:  {m5_f1:.4f}")
print(f"AUC-ROC:   {m5_auc:.4f}")


EVALUATION & COMPARISON

────────────────────────────────────────────────────────────────────────────────
[MODEL 1] LightGBM on Embeddings Only (TF-IDF + PhoBERT)
────────────────────────────────────────────────────────────────────────────────
Accuracy:  0.9051
Precision: 0.9002
Recall:    0.9051
F1 Score:  0.9011
AUC-ROC:   0.9272

────────────────────────────────────────────────────────────────────────────────
[MODEL 2] LightGBM on Combined Features (TF-IDF + PhoBERT + Hand-crafted)
────────────────────────────────────────────────────────────────────────────────
Accuracy:  0.9177
Precision: 0.9142
Recall:    0.9177
F1 Score:  0.9148
AUC-ROC:   0.9381

────────────────────────────────────────────────────────────────────────────────
[MODEL 3] Ensemble (LightGBM(embeddings) avg LightGBM(features))
────────────────────────────────────────────────────────────────────────────────
  Sub-model 3a (Embeddings): LightGBM on TF-IDF + PhoBERT
  Sub-model 3b (Features): LightGBM on Hand-crafted


In [10]:
print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORTS")
print("="*80)

print(f"\n[MODEL 1] LightGBM on Embeddings Classification Report:")
print(f"\n{classification_report(y_val, y_val_pred_m1, target_names=['Fake', 'Real'])}")

print(f"\n[MODEL 2] LightGBM on Combined Features Classification Report:")
print(f"\n{classification_report(y_val, y_val_pred_m2, target_names=['Fake', 'Real'])}")

if y_val_pred_m3 is not None:
    print(f"\n[MODEL 3] Ensemble Classification Report:")
    print(f"\n{classification_report(y_val, y_val_pred_m3, target_names=['Fake', 'Real'])}")

print(f"\n[MODEL 4] Ensemble on Embeddings Classification Report:")
print(f"\n{classification_report(y_val, y_val_pred_m4, target_names=['Fake', 'Real'])}")

print(f"\n[MODEL 5] Ensemble on Combined Classification Report:")
print(f"\n{classification_report(y_val, y_val_pred_m5, target_names=['Fake', 'Real'])}")


DETAILED CLASSIFICATION REPORTS

[MODEL 1] LightGBM on Embeddings Classification Report:

              precision    recall  f1-score   support

        Fake       0.93      0.96      0.94       393
        Real       0.77      0.63      0.69        81

    accuracy                           0.91       474
   macro avg       0.85      0.80      0.82       474
weighted avg       0.90      0.91      0.90       474


[MODEL 2] LightGBM on Combined Features Classification Report:

              precision    recall  f1-score   support

        Fake       0.94      0.97      0.95       393
        Real       0.81      0.68      0.74        81

    accuracy                           0.92       474
   macro avg       0.87      0.82      0.84       474
weighted avg       0.91      0.92      0.91       474


[MODEL 3] Ensemble Classification Report:

              precision    recall  f1-score   support

        Fake       0.93      0.97      0.95       393
        Real       0.81      0.67    

## 10. Summary & Comparison

In [11]:
print("\n" + "="*80)
print("COMPARISON SUMMARY TABLE")
print("="*80 + "\n")

if y_val_pred_m3 is not None:
    results_table = pd.DataFrame({
        'Model': [
            'Model 1: LightGBM (Emb)',
            'Model 2: LightGBM (All)',
            'Model 3: Ensemble (Emb+Features)',
            'Model 4: Ensemble (Emb)',
            'Model 5: Ensemble (All)'
        ],
        'Accuracy': [m1_accuracy, m2_accuracy, m3_accuracy, m4_accuracy, m5_accuracy],
        'Precision': [m1_precision, m2_precision, m3_precision, m4_precision, m5_precision],
        'Recall': [m1_recall, m2_recall, m3_recall, m4_recall, m5_recall],
        'F1 Score': [m1_f1, m2_f1, m3_f1, m4_f1, m5_f1],
        'AUC-ROC': [m1_auc, m2_auc, m3_auc, m4_auc, m5_auc]
    })
else:
    results_table = pd.DataFrame({
        'Model': [
            'Model 1: LightGBM (Emb)',
            'Model 2: LightGBM (All)',
            'Model 4: Ensemble (Emb)',
            'Model 5: Ensemble (All)'
        ],
        'Accuracy': [m1_accuracy, m2_accuracy, m4_accuracy, m5_accuracy],
        'Precision': [m1_precision, m2_precision, m4_precision, m5_precision],
        'Recall': [m1_recall, m2_recall, m4_recall, m5_recall],
        'F1 Score': [m1_f1, m2_f1, m4_f1, m5_f1],
        'AUC-ROC': [m1_auc, m2_auc, m4_auc, m5_auc]
    })

print(results_table.to_string(index=False))

print("\n" + "="*80)
print("ANALYSIS")
print("="*80)

best_idx = results_table['F1 Score'].idxmax()
best_model = results_table.iloc[best_idx]['Model']
best_f1 = results_table.iloc[best_idx]['F1 Score']

print(f"\n🏆 Best Model: {best_model}")
print(f"   F1 Score: {best_f1:.4f}")

print(f"\n📊 Model Comparison:")
print(f"   Model 1 F1: {m1_f1:.4f} (LightGBM on embeddings only)")
print(f"   Model 2 F1: {m2_f1:.4f} (LightGBM on all features)")
if y_val_pred_m3 is not None:
    print(f"   Model 3 F1: {m3_f1:.4f} (LightGBM(embeddings) avg LightGBM(features))")
print(f"   Model 4 F1: {m4_f1:.4f} (LightGBM avg CatBoost on embeddings)")
print(f"   Model 5 F1: {m5_f1:.4f} (LightGBM avg CatBoost on all features)")

print(f"\n📈 Model Architecture:")
print(f"   Model 1:  Embeddings only (TF-IDF + PhoBERT) → 1 LightGBM")
print(f"   Model 2:  All features (TF-IDF + PhoBERT + Hand-crafted) → 1 LightGBM")
print(f"   Model 3:  2 sub-models → avg predictions")
print(f"             • Sub-model 3a: Embeddings (TF-IDF + PhoBERT) → LightGBM")
print(f"             • Sub-model 3b: Hand-crafted → LightGBM")
print(f"   Model 4:  2 algorithms on embeddings → avg predictions")
print(f"             • Sub-model 4a: LightGBM on embeddings")
print(f"             • Sub-model 4b: CatBoost on embeddings")
print(f"   Model 5:  2 algorithms on all features → avg predictions")
print(f"             • Sub-model 5a: LightGBM on all features")
print(f"             • Sub-model 5b: CatBoost on all features")


COMPARISON SUMMARY TABLE

                           Model  Accuracy  Precision   Recall  F1 Score  AUC-ROC
         Model 1: LightGBM (Emb)  0.905063   0.900198 0.905063  0.901109 0.927245
         Model 2: LightGBM (All)  0.917722   0.914235 0.917722  0.914802 0.938083
Model 3: Ensemble (Emb+Features)  0.915612   0.911840 0.915612  0.912359 0.937643
         Model 4: Ensemble (Emb)  0.894515   0.890142 0.894515  0.891707 0.928690
         Model 5: Ensemble (All)  0.917722   0.917328 0.917722  0.917518 0.941099

ANALYSIS

🏆 Best Model: Model 5: Ensemble (All)
   F1 Score: 0.9175

📊 Model Comparison:
   Model 1 F1: 0.9011 (LightGBM on embeddings only)
   Model 2 F1: 0.9148 (LightGBM on all features)
   Model 3 F1: 0.9124 (LightGBM(embeddings) avg LightGBM(features))
   Model 4 F1: 0.8917 (LightGBM avg CatBoost on embeddings)
   Model 5 F1: 0.9175 (LightGBM avg CatBoost on all features)

📈 Model Architecture:
   Model 1:  Embeddings only (TF-IDF + PhoBERT) → 1 LightGBM
   Model 2:  All

## 11. Model Complexity & Computational Analysis

In [16]:
import pickle
import time
import numpy as np

print("\n" + "="*80)
print("COMPUTATIONAL EFFICIENCY ANALYSIS (F1 Score vs Complexity)")
print("="*80 + "\n")

# Dictionary to store models for FLOPs/params/memory calculation
models_dict = {
    '1': {'model': clf_model1, 'input_dim': X_train_embeddings_scaled.shape[1], 'name': 'LightGBM (Emb)'},
    '2': {'model': clf_model2, 'input_dim': X_train_scaled.shape[1], 'name': 'LightGBM (All)'},
    '3a': {'model': clf_model3a, 'input_dim': X_train_embeddings_scaled.shape[1], 'name': 'LightGBM (Emb)'},
    '3b': {'model': clf_model3b, 'input_dim': X_train_features_scaled.shape[1], 'name': 'LightGBM (Features)'},
    '4a': {'model': clf_model4a, 'input_dim': X_train_embeddings_scaled.shape[1], 'name': 'LightGBM (Emb)'},
    '4b': {'model': clf_model4b, 'input_dim': X_train_embeddings_scaled.shape[1], 'name': 'CatBoost (Emb)'},
    '5a': {'model': clf_model5a, 'input_dim': X_train_scaled.shape[1], 'name': 'LightGBM (All)'},
    '5b': {'model': clf_model5b, 'input_dim': X_train_scaled.shape[1], 'name': 'CatBoost (All)'},
}

complexity_data = []

for model_id, model_dict in models_dict.items():
    model = model_dict['model']
    input_dim = model_dict['input_dim']
    model_name = model_dict['name']
    
    # 1. Parameter counting
    if hasattr(model, 'booster_'):  # LightGBM
        n_estimators = model.n_estimators
        # Estimate: tree-based, approximately n_estimators * log(input_dim) nodes
        parameters = n_estimators * input_dim
    elif hasattr(model, 'tree_count_'):  # CatBoost
        n_estimators = model.tree_count_
        parameters = n_estimators * input_dim
    else:
        parameters = 0
    
    # 2. FLOPs calculation (tree-based models)
    if hasattr(model, 'booster_'):  # LightGBM
        avg_tree_depth = 7  # Default LightGBM depth
        flops_per_sample = n_estimators * avg_tree_depth * 2
    elif hasattr(model, 'tree_count_'):  # CatBoost
        avg_tree_depth = 6  # CatBoost usually runs deeper
        flops_per_sample = n_estimators * avg_tree_depth * 2
    else:
        flops_per_sample = 0
    
    # 3. Inference time (average of 10 runs on 100 random samples)
    try:
        sample_idx = np.random.choice(X_val.shape[0], min(100, X_val.shape[0]), replace=False)
        if model_id in ['1', '3a', '4a', '4b', '5a', '5b']:
            X_test = X_train_embeddings_scaled[sample_idx] if model_id in ['1', '3a', '4a', '4b'] else X_train_scaled[sample_idx]
        else:
            X_test = X_train_scaled[sample_idx]
        
        times = []
        for _ in range(10):
            start = time.time()
            _ = model.predict(X_test)
            times.append((time.time() - start) * 1000)  # ms
        inference_time = np.mean(times)
    except:
        inference_time = 0
    
    # 4. Memory footprint (model serialization size)
    try:
        model_bytes = pickle.dumps(model)
        memory_kb = len(model_bytes) / 1024
    except:
        memory_kb = 0
    
    complexity_data.append({
        'Model': f'Model {model_id}',
        'Type': 'LightGBM' if hasattr(model, 'booster_') else 'CatBoost',
        'Input Dims': input_dim,
        'Parameters': parameters,
        'FLOPs/Sample': flops_per_sample,
        'Inference (ms/100)': inference_time,
        'Memory (KB)': memory_kb
    })

# Create individual models table
complexity_df = pd.DataFrame(complexity_data)
print("TABLE 1: Individual Model Complexity Metrics")
print("-" * 120)
print(complexity_df[[col for col in complexity_df.columns if col != 'Type']].to_string(index=False))

# Ensemble summary
print("\n" + "="*80)
print("TABLE 2: Ensemble Models Summary")
print("-" * 80)

ensemble_summary = pd.DataFrame({
    'Ensemble Model': ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5'],
    'Sub-Models': [
        'LightGBM (Emb)',
        'LightGBM (All)',
        'LightGBM (Emb) + LightGBM (Features)',
        'LightGBM (Emb) + CatBoost (Emb)',
        'LightGBM (All) + CatBoost (All)'
    ],
    'Total Params': [
        complexity_data[0]['Parameters'],
        complexity_data[1]['Parameters'],
        complexity_data[2]['Parameters'] + complexity_data[3]['Parameters'],
        complexity_data[4]['Parameters'] + complexity_data[5]['Parameters'],
        complexity_data[6]['Parameters'] + complexity_data[7]['Parameters']
    ],
    'Total FLOPs/Sample': [
        complexity_data[0]['FLOPs/Sample'],
        complexity_data[1]['FLOPs/Sample'],
        complexity_data[2]['FLOPs/Sample'] + complexity_data[3]['FLOPs/Sample'],
        complexity_data[4]['FLOPs/Sample'] + complexity_data[5]['FLOPs/Sample'],
        complexity_data[6]['FLOPs/Sample'] + complexity_data[7]['FLOPs/Sample']
    ],
    'F1 Score': [m1_f1, m2_f1, m3_f1 if y_val_pred_m3 is not None else None, m4_f1, m5_f1],
    'AUC-ROC': [m1_auc, m2_auc, m3_auc if y_val_pred_m3 is not None else None, m4_auc, m5_auc]
})

ensemble_summary['Efficiency (F1/Params)'] = ensemble_summary['F1 Score'] / ensemble_summary['Total Params']
ensemble_summary['FLOPs Efficiency (F1/FLOPs)'] = ensemble_summary['F1 Score'] / ensemble_summary['Total FLOPs/Sample']

# Remove rows with None F1 scores for cleaner display
ensemble_summary_display = ensemble_summary.dropna(subset=['F1 Score'])
print(ensemble_summary_display[[
    'Ensemble Model', 'Total Params', 'Total FLOPs/Sample', 'F1 Score', 'AUC-ROC',
    'Efficiency (F1/Params)', 'FLOPs Efficiency (F1/FLOPs)'
]].to_string(index=False))

# Detailed breakdown
print("\n" + "="*80)
print("TABLE 3: Detailed Model Analysis")
print("-" * 80)

for idx, row in ensemble_summary[ensemble_summary['F1 Score'].notna()].iterrows():
    model_num = row['Ensemble Model']
    print(f"\n{model_num}: {row['Sub-Models']}")
    print(f"  F1 Score: {row['F1 Score']:.4f} | AUC-ROC: {row['AUC-ROC']:.4f}")
    print(f"  Parameters: {row['Total Params']:,} | FLOPs/Sample: {row['Total FLOPs/Sample']:,}")
    print(f"  Efficiency (F1/Params): {row['Efficiency (F1/Params)']:.6f}")
    print(f"  FLOPs Efficiency (F1/FLOPs): {row['FLOPs Efficiency (F1/FLOPs)']:.8f}")

# Best model by efficiency
print("\n" + "="*80)
best_by_params = ensemble_summary_display.loc[ensemble_summary_display['Efficiency (F1/Params)'].idxmax()]
best_by_flops = ensemble_summary_display.loc[ensemble_summary_display['FLOPs Efficiency (F1/FLOPs)'].idxmax()]

print(f"\n🏆 Best Model (F1/Parameters): {best_by_params['Ensemble Model']}")
print(f"   Efficiency: {best_by_params['Efficiency (F1/Params)']:.6f}")
print(f"   F1 Score: {best_by_params['F1 Score']:.4f}")

print(f"\n⚡ Best Model (F1/FLOPs): {best_by_flops['Ensemble Model']}")
print(f"   Efficiency: {best_by_flops['FLOPs Efficiency (F1/FLOPs)']:.8f}")
print(f"   F1 Score: {best_by_flops['F1 Score']:.4f}")

print("\n" + "="*80)
print("NOTES:")
print("-" * 80)
print("• FLOPs/Sample: Estimated floating-point operations per prediction sample")
print("  - LightGBM: n_estimators × avg_tree_depth × 2")
print("  - CatBoost: n_estimators × avg_tree_depth × 2")
print("• Parameters: Model weights and coefficients (estimated for tree-based models)")
print("• Inference Time: Average across 10 runs on 100 samples")
print("• Efficiency Ratios: F1 Score per unit of computational complexity")
print("• Ensemble FLOPs: Summed across all sub-models (represents total compute)")
print("="*80)


COMPUTATIONAL EFFICIENCY ANALYSIS (F1 Score vs Complexity)

TABLE 1: Individual Model Complexity Metrics
------------------------------------------------------------------------------------------------------------------------
   Model  Input Dims  Parameters  FLOPs/Sample  Inference (ms/100)  Memory (KB)
 Model 1        1668      333600          2800            1.129413   785.974609
 Model 2        1691      338200          2800            1.024628   787.495117
Model 3a        1668      333600          2800            0.990701   785.974609
Model 3b          23        4600          2800            0.000000   647.199219
Model 4a        1668      333600          2800            1.490331   785.974609
Model 4b        1668      333600          2400           31.460834   500.685547
Model 5a        1691      338200          2800            1.284313   787.495117
Model 5b        1691      338200          2400           31.287360   501.534180

TABLE 2: Ensemble Models Summary
-------------------